# TextZoom baseline check

1. Reproduce the **BICUBIC** rows of Table 4 (arXiv:2005.03341) with the migrated code (torch 2.5.1). This validates data, recognizers and the migration.
2. Time a few minutes of **TSRN** training to estimate the cost of the full 500 epochs.

Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# Closest stack to the SwinIR pruning project that installs on Python 3.13.
!nvidia-smi && python --version
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0

In [ ]:
REPO = '/content/TextZoomPruning'
%cd /content
!git clone -b migrate-torch2 https://github.com/Maouhoub/TextZoomPruning.git
%cd {REPO}
!pip install -q -r requirements.txt

import torch, torchvision, numpy
print(torch.__version__, torchvision.__version__, numpy.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Downloads are kept on Drive so later sessions skip them.
from google.colab import drive
drive.mount('/content/drive')

import os
STORE = '/content/drive/MyDrive/textzoom'
os.makedirs(STORE, exist_ok=True)

In [ ]:
%cd {STORE}
# TextZoom LMDB (official Google Drive folder from the TextZoom README)
![ -f .data_done ] || (gdown --folder https://drive.google.com/drive/folders/1WRVy-fC_KrembPkaI68uqQ9wyaptibMh -O data && touch .data_done)
# Pretrained recognizers, used as-is
![ -f aster.pth.tar ] || wget -q -O aster.pth.tar https://github.com/ayumiymk/aster.pytorch/releases/download/v1.0/demo.pth.tar
![ -f moran.pth ] || gdown 1IDvT51MXKSseDq3X57uPjOzeSYI09zip -O moran.pth
![ -f crnn.pth ] || wget -q -O crnn.pth "https://www.dropbox.com/s/dboqjk20qjkpta3/crnn.pth?dl=1"
!ls -la {STORE}

# LMDB reads from the Drive mount are slow, so work on a local copy.
!rm -rf /content/data && cp -r {STORE}/data /content/data
!find /content/data -name data.mdb

In [ ]:
# Point the config at the data and recognizer weights.
import glob, yaml

lmdb_dirs = sorted(os.path.dirname(p) for p in glob.glob('/content/data/**/data.mdb', recursive=True))
by_name = {os.path.basename(d): d for d in lmdb_dirs}
print(by_name)
TEST = {split: by_name[split] for split in ('easy', 'medium', 'hard')}

cfg_path = f'{REPO}/src/config/super_resolution.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['TRAIN']['train_data_dir'] = [by_name['train1'], by_name['train2']]
cfg['TRAIN']['VAL']['val_data_dir'] = [TEST['hard'], TEST['easy'], TEST['medium']]
cfg['TRAIN']['VAL']['rec_pretrained'] = f'{STORE}/aster.pth.tar'
cfg['TRAIN']['VAL']['moran_pretrained'] = f'{STORE}/moran.pth'
cfg['TRAIN']['VAL']['crnn_pretrained'] = f'{STORE}/crnn.pth'
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
print(open(cfg_path).read())

## 1. BICUBIC sanity check (no training)

Our accuracy should be within ~1 point of the paper. If ASTER/MORAN are off by more, the `align_corners` choice is the first suspect.

In [ ]:
import ast, subprocess

SPLITS = ('easy', 'medium', 'hard')
PAPER_ACC = {'aster': (64.7, 42.4, 31.2), 'moran': (60.6, 37.9, 30.8), 'crnn': (36.4, 21.1, 21.1)}  # Table 4
PAPER_PSNR = (22.35, 18.98, 19.39)  # Table 11

results = {}
for rec, paper_acc in PAPER_ACC.items():
    for split, paper in zip(SPLITS, paper_acc):
        run = subprocess.run(['python', 'main.py', '--arch', 'bicubic', '--test', '--rec', rec,
                              '--batch_size', '256', '--test_data_dir', TEST[split]],
                             cwd=f'{REPO}/src', capture_output=True, text=True)
        lines = run.stdout.strip().splitlines()
        if run.returncode != 0 or not lines or not lines[-1].startswith('{'):
            print(run.stdout[-2000:], run.stderr[-4000:])
            raise RuntimeError(f'{rec}/{split} failed')
        result = ast.literal_eval(lines[-1])
        results[(rec, split)] = result
        acc = result['accuracy'][split] * 100
        print(f'{rec:6s} {split:7s} acc {acc:5.1f} (paper {paper:5.1f})  psnr {result["psnr_avg"]:.2f}')

print('paper PSNR easy/medium/hard:', PAPER_PSNR)

## 2. Time TSRN training (15 minutes)

Prints every 10 iterations. One epoch is 67 iterations at batch 256 (17,367 images). Estimate: seconds per iteration × 67 × 500 epochs. The paper's batch of 512 was split over 4 GPUs (128 each), so 512 does not fit on one T4. Validation (ASTER on the 3 test sets) runs every 200 iterations and is included. If you get an out-of-memory error, use `--batch_size 128`.

In [ ]:
cfg = yaml.safe_load(open(cfg_path))
cfg['TRAIN']['displayInterval'] = 10
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)

%cd {REPO}/src
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True timeout 15m python main.py --arch tsrn --batch_size 256 --STN --mask --gradient --vis_dir tsrn_timing